# 09. Trading Signal Generation & Backtesting

**Objective:** Generate actionable trading signals from NLP analysis and backtest performance

**Signal Components:**
- **Sentiment Signal** (35%): FinBERT sentiment momentum and extremity
- **Event Impact** (25%): High-impact event detection and scoring
- **News Velocity** (20%): Article frequency and burst detection
- **Entity Prominence** (15%): Company mention frequency and co-mentions
- **Topic Relevance** (5%): Topic dominance and shifts

**Signal Categories:**
- **Strong Buy** (>0.7): High confidence positive signal
- **Buy** (0.4-0.7): Moderate positive signal
- **Neutral** (-0.4 to 0.4): No clear direction
- **Sell** (-0.7 to -0.4): Moderate negative signal
- **Strong Sell** (<-0.7): High confidence negative signal

In [2]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Backtesting
from sklearn.metrics import accuracy_score
from scipy import stats

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Setup Paths and Configuration

In [3]:
# Setup paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / '01_DATA'
FEATURES_DIR = DATA_DIR / 'features'
RESULTS_DIR = BASE_DIR / '03_RESULTS'
OUTPUTS_DIR = RESULTS_DIR / 'outputs'
VIZ_DIR = RESULTS_DIR / 'visualizations'
CONFIG_DIR = BASE_DIR / '06_CONFIG'

# Create directories
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Load configuration
with open(CONFIG_DIR / 'framework_config.json', 'r') as f:
    config = json.load(f)

print(f" Base directory: {BASE_DIR}")
print(f" Results directory: {RESULTS_DIR}")
print(f" Configuration loaded")

 Base directory: c:\Users\HARSHIT\Desktop\p\nlp analysis
 Results directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS
 Configuration loaded


## 2. Load Final Feature Matrix

In [4]:
# Load final feature matrix from notebook 07
df = pd.read_csv(FEATURES_DIR / 'final_feature_matrix.csv')
df['date'] = pd.to_datetime(df['date'])

# Sort by ticker and date
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

print(f" Loaded {len(df):,} articles")
print(f" Date range: {df['date'].min()} to {df['date'].max()}")
print(f" Companies: {df['ticker'].nunique()}")
print(f" Features: {df.shape[1]}")

df.head()

 Loaded 63 articles
 Date range: 2025-09-26 07:00:00 to 2026-01-18 12:19:48
 Companies: 19
 Features: 93


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,...,topic_stability,hour,day_of_week,is_weekend,is_market_hours,days_since_last,recency_score,sentiment_lag_1,sentiment_lag_3,sentiment_lag_7
0,2026-01-14 08:02:00,This Unstoppable Stock Has 4 Catalysts to Fuel...,This Unstoppable Stock Has 4 Catalysts to Fuel...,Google News,https://news.google.com/rss/articles/CBMimAFBV...,AAPL,NaN,AAPL stock news,157,25,...,0.000000,8,2,0,0,0.000000,0.869966,NaN,NaN,NaN
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,...,0.000000,12,6,1,1,4.175833,0.999894,-0.5719,NaN,NaN
2,2025-12-27 08:00:00,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,Google News,https://news.google.com/rss/articles/CBMiiwFBV...,ABT,NaN,ABT stock news,152,21,...,0.000000,8,5,1,0,0.000000,0.477425,NaN,NaN,NaN
3,2025-12-31 08:00:00,Here is What to Know Beyond Why Abbott Laborat...,Here is What to Know Beyond Why Abbott Laborat...,Google News,https://news.google.com/rss/articles/CBMiiAFBV...,ABT,NaN,ABT stock news,102,15,...,0.500000,8,2,0,0,4.000000,0.545521,-0.3818,NaN,NaN
4,2026-01-15 16:55:56,Dividend King Abbott Shows Why 52 Consecutive ...,Dividend King Abbott Shows Why 52 Consecutive ...,Google News,https://news.google.com/rss/articles/CBMi2gFBV...,ABT,NaN,ABT stock news,124,17,...,0.666667,16,3,0,1,15.372176,0.910640,0.0000,NaN,NaN


## 3. Calculate Multi-Factor Trading Signals

In [5]:
print(" Calculating trading signals...\n")

def calculate_trading_signal(row):
    """
    Calculate composite trading signal from multiple factors
    
    Weights:
    - Sentiment: 35%
    - Event Impact: 25%
    - News Velocity: 20%
    - Entity Prominence: 15%
    - Topic Relevance: 5%
    """
    
    # 1. Sentiment signal (35%)
    sentiment_score = row.get('finbert_sentiment', 0)
    sentiment_momentum = row.get('sentiment_momentum', 0)
    sentiment_extremity = row.get('sentiment_extremity', 0)
    
    sentiment_signal = (
        sentiment_score * 0.5 +
        sentiment_momentum * 0.3 +
        sentiment_extremity * np.sign(sentiment_score) * 0.2
    )
    
    # 2. Event impact signal (25%)
    event_impact = row.get('event_impact_score', 0)
    has_high_impact = row.get('has_high_impact_event', 0)
    
    event_signal = (
        event_impact * 0.7 +
        has_high_impact * 0.3
    ) * np.sign(sentiment_score)  # Direction from sentiment
    
    # 3. News velocity signal (20%)
    news_burst = row.get('news_burst', 0)
    article_count = row.get('article_count_7d', 0)
    news_acceleration = row.get('news_acceleration', 0)
    
    velocity_signal = (
        news_burst * 0.4 +
        min(article_count / 50, 1.0) * 0.4 +
        np.sign(news_acceleration) * 0.2
    ) * np.sign(sentiment_score)
    
    # 4. Entity prominence signal (15%)
    entity_density = row.get('entity_density', 0)
    entity_freq = row.get('entity_freq_7d', 0)
    
    entity_signal = (
        min(entity_density * 100, 1.0) * 0.6 +
        min(entity_freq / 100, 1.0) * 0.4
    ) * np.sign(sentiment_score)
    
    # 5. Topic relevance signal (5%)
    topic_dominance = row.get('topic_dominance', 0)
    
    topic_signal = topic_dominance * np.sign(sentiment_score)
    
    # Combine with weights
    composite_signal = (
        sentiment_signal * 0.35 +
        event_signal * 0.25 +
        velocity_signal * 0.20 +
        entity_signal * 0.15 +
        topic_signal * 0.05
    )
    
    # Normalize to [-1, 1]
    composite_signal = np.clip(composite_signal, -1.0, 1.0)
    
    return composite_signal

# Calculate signals
df['trading_signal'] = df.apply(calculate_trading_signal, axis=1)

# Categorize signals
def categorize_signal(signal):
    if signal > 0.7:
        return 'Strong Buy'
    elif signal > 0.4:
        return 'Buy'
    elif signal > -0.4:
        return 'Neutral'
    elif signal > -0.7:
        return 'Sell'
    else:
        return 'Strong Sell'

df['signal_category'] = df['trading_signal'].apply(categorize_signal)

print(" Trading signals calculated\n")
print(" Signal distribution:")
print(df['signal_category'].value_counts())
print(f"\n{df['signal_category'].value_counts(normalize=True) * 100}")

 Calculating trading signals...

 Trading signals calculated

 Signal distribution:
signal_category
Neutral        42
Strong Sell    19
Buy             1
Sell            1
Name: count, dtype: int64

signal_category
Neutral        66.666667
Strong Sell    30.158730
Buy             1.587302
Sell            1.587302
Name: proportion, dtype: float64


## 4. Signal Statistics by Company

In [6]:
# Aggregate signals by company
company_signals = df.groupby('ticker').agg({
    'trading_signal': ['mean', 'std', 'min', 'max'],
    'signal_category': lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Neutral',
    'date': 'count'
}).reset_index()

company_signals.columns = ['ticker', 'avg_signal', 'signal_std', 'min_signal', 'max_signal', 'dominant_category', 'article_count']
company_signals = company_signals.sort_values('avg_signal', ascending=False)

print(" Top 20 Companies by Average Signal:")
print("=" * 100)
print(company_signals.head(20).to_string(index=False))

# Save
company_signals.to_csv(OUTPUTS_DIR / 'company_trading_signals.csv', index=False)
print(f"\n Saved to {OUTPUTS_DIR / 'company_trading_signals.csv'}")

 Top 20 Companies by Average Signal:
ticker  avg_signal  signal_std  min_signal  max_signal dominant_category  article_count
  ASML    0.307478         NaN    0.307478    0.307478           Neutral              2
   AXP    0.209113         NaN    0.209113    0.209113           Neutral              2
   BBD    0.179752         NaN    0.179752    0.179752           Neutral              2
  BABA    0.154707    0.266922   -0.249330    0.404769           Neutral              6
   APA    0.130340    0.174427   -0.008106    0.326250           Neutral              4
   AVB    0.112569    0.212833   -0.037926    0.263065           Neutral              3
   BAC    0.104399    0.356009   -0.405843    0.387927           Neutral              5
    BA    0.096565    0.102069   -0.021241    0.158500           Neutral              4
  AAPL    0.060049         NaN    0.060049    0.060049           Neutral              2
   AMD    0.041524    0.226325   -0.219536    0.182494           Neutral           

## 5. Visualize Trading Signals

In [7]:
# Signal distribution histogram
fig = px.histogram(
    df,
    x='trading_signal',
    nbins=50,
    title='Trading Signal Distribution',
    labels={'trading_signal': 'Signal Strength', 'count': 'Frequency'},
    color_discrete_sequence=['steelblue']
)

fig.add_vline(x=0.7, line_dash="dash", line_color="green", annotation_text="Strong Buy")
fig.add_vline(x=0.4, line_dash="dash", line_color="lightgreen", annotation_text="Buy")
fig.add_vline(x=-0.4, line_dash="dash", line_color="lightcoral", annotation_text="Sell")
fig.add_vline(x=-0.7, line_dash="dash", line_color="red", annotation_text="Strong Sell")

fig.update_layout(height=500)
fig.write_html(VIZ_DIR / 'signal_distribution.html')
fig.show()

# Signal category pie chart
fig2 = px.pie(
    df['signal_category'].value_counts().reset_index(),
    values='count',
    names='signal_category',
    title='Signal Category Distribution',
    color='signal_category',
    color_discrete_map={
        'Strong Buy': 'darkgreen',
        'Buy': 'lightgreen',
        'Neutral': 'gray',
        'Sell': 'lightcoral',
        'Strong Sell': 'darkred'
    }
)

fig2.write_html(VIZ_DIR / 'signal_categories_pie.html')
fig2.show()

## 6. Signals Over Time

In [8]:
# Average signal over time
df['year_month'] = df['date'].dt.to_period('M').astype(str)

signal_timeline = df.groupby('year_month').agg({
    'trading_signal': ['mean', 'std'],
    'date': 'count'
}).reset_index()

signal_timeline.columns = ['year_month', 'avg_signal', 'signal_std', 'article_count']

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=signal_timeline['year_month'],
    y=signal_timeline['avg_signal'],
    mode='lines+markers',
    name='Average Signal',
    line=dict(color='steelblue', width=2),
    error_y=dict(
        type='data',
        array=signal_timeline['signal_std'],
        visible=True
    )
))

fig.add_hline(y=0, line_dash="dash", line_color="gray", annotation_text="Neutral")
fig.add_hline(y=0.4, line_dash="dot", line_color="green", annotation_text="Buy Threshold")
fig.add_hline(y=-0.4, line_dash="dot", line_color="red", annotation_text="Sell Threshold")

fig.update_layout(
    title='Trading Signals Over Time (Monthly Average)',
    xaxis_title='Month',
    yaxis_title='Average Signal',
    height=500,
    xaxis_tickangle=-45,
    hovermode='x unified'
)

fig.write_html(VIZ_DIR / 'signals_over_time.html')
fig.show()

## 7. Signal Strength by Event Type

In [9]:
# Average signal by primary event
if 'primary_event' in df.columns:
    event_signals = df.groupby('primary_event').agg({
        'trading_signal': 'mean',
        'date': 'count'
    }).reset_index()
    
    event_signals.columns = ['event', 'avg_signal', 'count']
    event_signals = event_signals[event_signals['count'] >= 10]  # Filter rare events
    event_signals = event_signals.sort_values('avg_signal', ascending=False)
    
    print(" Average Signal by Event Type:")
    print("=" * 70)
    print(event_signals.to_string(index=False))
    
    # Visualize
    fig = px.bar(
        event_signals,
        x='event',
        y='avg_signal',
        title='Average Trading Signal by Event Type',
        labels={'avg_signal': 'Average Signal', 'event': 'Event Type'},
        color='avg_signal',
        color_continuous_scale='RdYlGn',
        color_continuous_midpoint=0
    )
    
    fig.update_layout(height=600, xaxis_tickangle=-45)
    fig.write_html(VIZ_DIR / 'signals_by_event.html')
    fig.show()

 Average Signal by Event Type:
   event  avg_signal  count
NO_EVENT    0.057468     62


## 8. Backtest Framework

In [10]:
print(" Setting up backtesting framework...\n")

# Create synthetic returns for demonstration
# In production, you would load actual stock price data
def generate_synthetic_returns(signal, noise_level=0.02):
    """
    Generate synthetic returns based on signal
    In production: Replace with actual stock returns data
    """
    # Base return follows signal direction
    base_return = signal * 0.05  # Signal influence
    
    # Add market noise
    noise = np.random.normal(0, noise_level)
    
    return base_return + noise

np.random.seed(42)
df['forward_return'] = df['trading_signal'].apply(generate_synthetic_returns)


 Setting up backtesting framework...

 Backtesting framework ready
 Note: Using synthetic returns for demonstration
   In production: Replace with actual stock price data


## 9. Signal Performance Analysis

In [11]:
# Analyze signal effectiveness
df['signal_correct'] = ((df['trading_signal'] > 0) & (df['forward_return'] > 0)) | \
                       ((df['trading_signal'] < 0) & (df['forward_return'] < 0))

# Overall win rate
win_rate = df['signal_correct'].mean()

# Win rate by signal category
category_performance = df.groupby('signal_category').agg({
    'signal_correct': 'mean',
    'forward_return': ['mean', 'std'],
    'date': 'count'
}).reset_index()

category_performance.columns = ['category', 'win_rate', 'avg_return', 'return_std', 'count']

print(" Signal Performance Analysis:")
print("=" * 80)
print(f"Overall Win Rate: {win_rate:.4f} ({win_rate*100:.2f}%)")
print(f"\nPerformance by Signal Category:")
print(category_performance.to_string(index=False))

# Save
category_performance.to_csv(OUTPUTS_DIR / 'signal_performance_by_category.csv', index=False)
print(f"\n Saved to {OUTPUTS_DIR / 'signal_performance_by_category.csv'}")

 Signal Performance Analysis:
Overall Win Rate: 0.5079 (50.79%)

Performance by Signal Category:
   category  win_rate  avg_return  return_std  count
        Buy  1.000000    0.032472         NaN      1
    Neutral  0.714286    0.000853     0.02456     42
       Sell  1.000000   -0.026476         NaN      1
Strong Sell  0.000000         NaN         NaN     19

 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\signal_performance_by_category.csv


## 10. Risk-Adjusted Metrics

In [12]:
# Calculate Sharpe Ratio (annualized)
returns = df['forward_return'].values
sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(252)  # Annualized

# Maximum drawdown
cumulative_returns = (1 + df['forward_return']).cumprod()
running_max = cumulative_returns.expanding().max()
drawdown = (cumulative_returns - running_max) / running_max
max_drawdown = drawdown.min()

# Sortino Ratio (downside deviation)
downside_returns = returns[returns < 0]
downside_std = downside_returns.std()
sortino_ratio = (returns.mean() / downside_std) * np.sqrt(252) if downside_std > 0 else 0

# Calmar Ratio
calmar_ratio = (returns.mean() * 252) / abs(max_drawdown) if max_drawdown != 0 else 0

print(" Risk-Adjusted Performance Metrics:")
print("=" * 70)
print(f"Sharpe Ratio: {sharpe_ratio:.4f}")
print(f"Sortino Ratio: {sortino_ratio:.4f}")
print(f"Calmar Ratio: {calmar_ratio:.4f}")
print(f"Maximum Drawdown: {max_drawdown:.4f} ({max_drawdown*100:.2f}%)")
print(f"\nAverage Return: {returns.mean():.4f} ({returns.mean()*100:.2f}%)")
print(f"Return Volatility: {returns.std():.4f} ({returns.std()*100:.2f}%)")

# Compare to benchmarks
print(f"\n Benchmark Comparison:")
print(f"  Win Rate: {win_rate:.4f} (Target: 0.563, Benchmark: 0.52-0.58)")
print(f"  Sharpe Ratio: {sharpe_ratio:.4f} (Target: 1.87, Benchmark: 1.2-1.8)")

if win_rate >= 0.563:
    print(f"   Win rate EXCEEDS target")
elif win_rate >= 0.52:
    print(f"   Win rate meets industry benchmark")
else:
    print(f"   Win rate below benchmark")

if sharpe_ratio >= 1.87:
    print(f"   Sharpe ratio EXCEEDS target")
elif sharpe_ratio >= 1.2:
    print(f"   Sharpe ratio meets industry benchmark")
else:
    print(f"   Sharpe ratio below benchmark")

 Risk-Adjusted Performance Metrics:
Sharpe Ratio: nan
Sortino Ratio: nan
Calmar Ratio: nan
Maximum Drawdown: -0.1317 (-13.17%)

Average Return: nan (nan%)
Return Volatility: nan (nan%)

 Benchmark Comparison:
  Win Rate: 0.5079 (Target: 0.563, Benchmark: 0.52-0.58)
  Sharpe Ratio: nan (Target: 1.87, Benchmark: 1.2-1.8)
   Win rate below benchmark
   Sharpe ratio below benchmark


## 11. Cumulative Returns Visualization

In [13]:
# Calculate cumulative returns over time
df_sorted = df.sort_values('date').reset_index(drop=True)
df_sorted['cumulative_return'] = (1 + df_sorted['forward_return']).cumprod() - 1

# Plot
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_sorted['date'],
    y=df_sorted['cumulative_return'] * 100,
    mode='lines',
    name='Strategy Returns',
    line=dict(color='steelblue', width=2)
))

fig.update_layout(
    title='Cumulative Strategy Returns Over Time',
    xaxis_title='Date',
    yaxis_title='Cumulative Return (%)',
    height=500,
    hovermode='x unified'
)

fig.write_html(VIZ_DIR / 'cumulative_returns.html')
fig.show()

## 12. Signal Factor Contribution Analysis

In [14]:
# Analyze contribution of each factor to signal performance
factor_analysis = pd.DataFrame({
    'Factor': [
        'Sentiment (35%)',
        'Event Impact (25%)',
        'News Velocity (20%)',
        'Entity Prominence (15%)',
        'Topic Relevance (5%)'
    ],
    'Weight': [0.35, 0.25, 0.20, 0.15, 0.05],
    'Description': [
        'FinBERT sentiment + momentum + extremity',
        'Event impact score + high-impact detection',
        'Article count + burst detection + acceleration',
        'Entity density + frequency',
        'Topic dominance'
    ]
})

print(" Signal Factor Contribution:")
print("=" * 100)
print(factor_analysis.to_string(index=False))

# Visualize factor weights
fig = px.pie(
    factor_analysis,
    values='Weight',
    names='Factor',
    title='Trading Signal Factor Weights',
    hole=0.3
)

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.write_html(VIZ_DIR / 'signal_factor_weights.html')
fig.show()

# Save
factor_analysis.to_csv(OUTPUTS_DIR / 'signal_factor_weights.csv', index=False)
print(f"\n Saved to {OUTPUTS_DIR / 'signal_factor_weights.csv'}")

 Signal Factor Contribution:
                 Factor  Weight                                    Description
        Sentiment (35%)    0.35       FinBERT sentiment + momentum + extremity
     Event Impact (25%)    0.25     Event impact score + high-impact detection
    News Velocity (20%)    0.20 Article count + burst detection + acceleration
Entity Prominence (15%)    0.15                     Entity density + frequency
   Topic Relevance (5%)    0.05                                Topic dominance



 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\signal_factor_weights.csv


## 13. Top Trading Signals (Actionable)

In [15]:
# Get recent strong signals
recent_date = df['date'].max() - timedelta(days=30)
recent_signals = df[df['date'] >= recent_date].copy()

# Top buy signals
top_buys = recent_signals[recent_signals['signal_category'].isin(['Strong Buy', 'Buy'])].nlargest(20, 'trading_signal')

# Top sell signals
top_sells = recent_signals[recent_signals['signal_category'].isin(['Strong Sell', 'Sell'])].nsmallest(20, 'trading_signal')

print(" Top 20 BUY Signals (Last 30 Days):")
print("=" * 100)
for idx, row in top_buys.iterrows():
    print(f"{row['ticker']:8s} | {row['date'].date()} | Signal: {row['trading_signal']:.3f} | "
          f"Category: {row['signal_category']:12s} | Sentiment: {row['finbert_sentiment']:.3f}")

print(f"\n Top 20 SELL Signals (Last 30 Days):")
print("=" * 100)
for idx, row in top_sells.iterrows():
    print(f"{row['ticker']:8s} | {row['date'].date()} | Signal: {row['trading_signal']:.3f} | "
          f"Category: {row['signal_category']:12s} | Sentiment: {row['finbert_sentiment']:.3f}")

# Save actionable signals
top_buys[['ticker', 'date', 'title', 'trading_signal', 'signal_category', 'finbert_sentiment', 'event_impact_score']].to_csv(
    OUTPUTS_DIR / 'top_buy_signals.csv', index=False
)
top_sells[['ticker', 'date', 'title', 'trading_signal', 'signal_category', 'finbert_sentiment', 'event_impact_score']].to_csv(
    OUTPUTS_DIR / 'top_sell_signals.csv', index=False
)

print(f"\n Saved actionable signals:")
print(f"  - {OUTPUTS_DIR / 'top_buy_signals.csv'}")
print(f"  - {OUTPUTS_DIR / 'top_sell_signals.csv'}")

 Top 20 BUY Signals (Last 30 Days):
BABA     | 2026-01-17 | Signal: 0.405 | Category: Buy          | Sentiment: 0.772

 Top 20 SELL Signals (Last 30 Days):
BAC      | 2026-01-14 | Signal: -0.406 | Category: Sell         | Sentiment: -0.637
AAPL     | 2026-01-14 | Signal: nan | Category: Strong Sell  | Sentiment: -0.572
ABT      | 2025-12-27 | Signal: nan | Category: Strong Sell  | Sentiment: -0.382
ACN      | 2026-01-14 | Signal: nan | Category: Strong Sell  | Sentiment: 0.477
ADBE     | 2026-01-13 | Signal: nan | Category: Strong Sell  | Sentiment: -0.420
AEP      | 2025-12-27 | Signal: nan | Category: Strong Sell  | Sentiment: 0.382
AMD      | 2026-01-15 | Signal: nan | Category: Strong Sell  | Sentiment: 0.681
AMGN     | 2026-01-16 | Signal: nan | Category: Strong Sell  | Sentiment: -0.153
AMZN     | 2026-01-14 | Signal: nan | Category: Strong Sell  | Sentiment: 0.178
ASML     | 2026-01-15 | Signal: nan | Category: Strong Sell  | Sentiment: 0.000
AVGO     | 2026-01-14 | Signal: nan 

## 14. Save Final Trading Signals

In [ ]:
# Save complete trading signals dataset with available columns
trading_signals_df = df[[
    'date', 'ticker', 'title', 'text',
    'finbert_sentiment', 'finbert_label', 'finbert_score',
    'event_impact_score', 'num_events', 'primary_event',
    'entity_density', 'entity_freq_7d', 'entity_diversity',
    'topic_dominance', 'dominant_topic', 'topic_keywords',
    'sentiment_momentum', 'sentiment_extremity',
    'trading_signal', 'signal_category',
    'forward_return', 'signal_correct'
]].copy()

trading_signals_df.to_csv(OUTPUTS_DIR / 'trading_signals_complete.csv', index=False)

print(f"Complete trading signals saved to {OUTPUTS_DIR / 'trading_signals_complete.csv'}")
print(f"Dataset shape: {trading_signals_df.shape}")
print(f"   Rows: {trading_signals_df.shape[0]:,} signals")
print(f"   Columns: {trading_signals_df.shape[1]} features")

trading_signals_df.head()

  Complete trading signals saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\trading_signals_complete.csv
  Dataset shape: (63, 22)
   Rows: 63 signals
   Columns: 22 features


,date,ticker,title,text,finbert_sentiment,finbert_label,finbert_score,event_impact_score,num_events,primary_event,...,entity_diversity,topic_dominance,dominant_topic,topic_keywords,sentiment_momentum,sentiment_extremity,trading_signal,signal_category,forward_return,signal_correct
0,2026-01-14 08:02:00,AAPL,This Unstoppable Stock Has 4 Catalysts to Fuel...,This Unstoppable Stock Has 4 Catalysts to Fuel...,-0.5719,negative,0.5719,0.0,0,NO_EVENT,...,0.666667,0.178254,1,"finance, yahoo, price",NaN,0.5719,NaN,Strong Sell,NaN,False
1,2026-01-18 12:15:12,AAPL,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,0.0000,neutral,0.0000,0.0,0,NO_EVENT,...,0.333333,0.725714,4,"price, finance, yahoo",0.5719,0.0000,0.060049,Neutral,0.000237,True
2,2025-12-27 08:00:00,ABT,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,Is Weakness In Abbott Laboratories (NYSE:ABT) ...,-0.3818,negative,0.3818,0.0,0,NO_EVENT,...,0.333333,0.800899,1,"finance, yahoo, price",NaN,0.3818,NaN,Strong Sell,NaN,False
3,2025-12-31 08:00:00,ABT,Here is What to Know Beyond Why Abbott Laborat...,Here is What to Know Beyond Why Abbott Laborat...,0.0000,neutral,0.0000,0.0,0,NO_EVENT,...,0.333333,0.800899,1,"finance, yahoo, price",0.3818,0.0000,0.040089,Neutral,0.032465,True
4,2026-01-15 16:55:56,ABT,Dividend King Abbott Shows Why 52 Consecutive ...,Dividend King Abbott Shows Why 52 Consecutive ...,-0.6348,negative,0.6348,0.0,0,NO_EVENT,...,0.666667,0.178254,1,"finance, yahoo, price",-0.6348,0.6348,-0.330093,Neutral,-0.021188,True


In [18]:
# Check available columns
print("Available columns in dataframe:")
print(df.columns.tolist())
print(f"\nTotal columns: {len(df.columns)}")

Available columns in dataframe:
['date', 'title', 'text', 'source', 'url', 'ticker', 'publisher', 'query', 'article_length', 'word_count', 'collection_date', 'finbert_label', 'finbert_score', 'finbert_sentiment', 'vader_compound', 'vader_label', 'textblob_polarity', 'textblob_label', 'date_only', 'spacy_entities_json', 'bert_entities_json', 'all_entities_json', 'num_entities', 'num_orgs', 'num_persons', 'num_locations', 'num_events', 'primary_event', 'primary_event_id', 'year_month', 'event_impact_score', 'impact_category', 'detected_events_json', 'event_names_json', 'event_ids_json', 'has_EARNINGS_ANNOUNCEMENT', 'has_MERGER_ACQUISITION', 'has_PRODUCT_LAUNCH', 'has_REGULATORY_ACTION', 'has_EXECUTIVE_CHANGE', 'has_LEGAL_ISSUE', 'has_PARTNERSHIP_DEAL', 'has_MARKET_DISRUPTION', 'has_DIVIDEND_ANNOUNCEMENT', 'has_RESTRUCTURING', 'has_ANALYST_RATING', 'has_ECONOMIC_INDICATOR', 'dominant_topic', 'topic_keywords', 'topic_distribution_json', 'topic_0_prob', 'topic_1_prob', 'topic_2_prob', 'topi

## 15. Final Summary Report

In [21]:
# Create comprehensive summary
summary = {
    'dataset': {
        'total_articles': len(df),
        'date_range': {
            'start': str(df['date'].min()),
            'end': str(df['date'].max())
        },
        'companies': int(df['ticker'].nunique())
    },
    'signal_distribution': dict(df['signal_category'].value_counts()),
    'performance_metrics': {
        'win_rate': float(win_rate),
        'sharpe_ratio': float(sharpe_ratio),
        'sortino_ratio': float(sortino_ratio),
        'calmar_ratio': float(calmar_ratio),
        'max_drawdown': float(max_drawdown),
        'avg_return': float(returns.mean()),
        'return_volatility': float(returns.std())
    },
    'signal_factors': {
        'sentiment_weight': 0.35,
        'event_impact_weight': 0.25,
        'news_velocity_weight': 0.20,
        'entity_prominence_weight': 0.15,
        'topic_relevance_weight': 0.05
    },
    'benchmark_comparison': {
        'win_rate': {
            'achieved': float(win_rate),
            'target': 0.563,
            'benchmark_range': '52-58%',
            'meets_target': bool(win_rate >= 0.563)
        },
        'sharpe_ratio': {
            'achieved': float(sharpe_ratio),
            'target': 1.87,
            'benchmark_range': '1.2-1.8',
            'meets_target': bool(sharpe_ratio >= 1.87)
        }
    },
    'top_companies': {
        'most_bullish': company_signals.head(5)[['ticker', 'avg_signal']].to_dict('records'),
        'most_bearish': company_signals.tail(5)[['ticker', 'avg_signal']].to_dict('records')
    }
}

# Save summary
with open(OUTPUTS_DIR / 'trading_signals_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(" Trading Signal Generation Summary")
print("=" * 70)
print(f"Total signals generated: {summary['dataset']['total_articles']:,}")
print(f"Companies covered: {summary['dataset']['companies']:,}")
print(f"\nSignal distribution:")
for category, count in summary['signal_distribution'].items():
    print(f"  {category}: {count:,}")
print(f"\nPerformance metrics:")
print(f"  Win Rate: {summary['performance_metrics']['win_rate']:.4f} ({summary['performance_metrics']['win_rate']*100:.2f}%)")
print(f"  Sharpe Ratio: {summary['performance_metrics']['sharpe_ratio']:.4f}")
print(f"  Sortino Ratio: {summary['performance_metrics']['sortino_ratio']:.4f}")
print(f"  Max Drawdown: {summary['performance_metrics']['max_drawdown']:.4f} ({summary['performance_metrics']['max_drawdown']*100:.2f}%)")

print(f"\n Benchmark comparison:")
print(f"  Win Rate: {' MEETS TARGET' if summary['benchmark_comparison']['win_rate']['meets_target'] else ' BELOW TARGET'}")
print(f"  Sharpe Ratio: {' MEETS TARGET' if summary['benchmark_comparison']['sharpe_ratio']['meets_target'] else ' BELOW TARGET'}")

print(f"\n Summary saved to {OUTPUTS_DIR / 'trading_signals_summary.json'}")
print("\n Trading Signal Generation & Backtesting complete!")
print("\n Output files:")
print(f"  - {OUTPUTS_DIR / 'trading_signals_complete.csv'}")
print(f"  - {OUTPUTS_DIR / 'company_trading_signals.csv'}")
print(f"  - {OUTPUTS_DIR / 'top_buy_signals.csv'}")
print(f"  - {OUTPUTS_DIR / 'top_sell_signals.csv'}")
print(f"  - {OUTPUTS_DIR / 'signal_performance_by_category.csv'}")
print(f"  - {VIZ_DIR / 'signal_distribution.html'}")
print(f"  - {VIZ_DIR / 'signals_over_time.html'}")
print(f"  - {VIZ_DIR / 'cumulative_returns.html'}")
print(f"  - {VIZ_DIR / 'signal_factor_weights.html'}")


 Trading Signal Generation Summary
Total signals generated: 63
Companies covered: 19

Signal distribution:
  Neutral: 42
  Strong Sell: 19
  Buy: 1
  Sell: 1

Performance metrics:
  Win Rate: 0.5079 (50.79%)
  Sharpe Ratio: nan
  Sortino Ratio: nan
  Max Drawdown: -0.1317 (-13.17%)

 Benchmark comparison:
  Win Rate:  BELOW TARGET
  Sharpe Ratio:  BELOW TARGET

 Summary saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\trading_signals_summary.json

 Trading Signal Generation & Backtesting complete!

 Output files:
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\trading_signals_complete.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\company_trading_signals.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\top_buy_signals.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\top_sell_signals.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\signal_performance_by_category.csv
  - c:\Users\H